# Trabalho Prático III: Recomendador de Disciplinas com NLP e Busca Semântica
**Disciplina:** Fundamentos de Inteligência Artificial (FIA) - Engenharia de Sistemas (UFMG)

Este notebook é **100% autossuficiente**, ou seja, ele baixa todos os dados necessários direto do repositório oficial no GitHub e carrega todas as funções na própria memória, sem necessidade de importar scripts externos.

In [ ]:
# 1. Download dos dados e Instalação de Bibliotecas Necessárias
!pip install -q sentence-transformers pandas numpy torch
!mkdir -p data
!wget -qO data/disciplinas_engsis.json https://raw.githubusercontent.com/AndreRezende1999/tp3-ufmg-recomendador/main/data/disciplinas_engsis.json
!wget -qO data/dificuldade_engsis.csv https://raw.githubusercontent.com/AndreRezende1999/tp3-ufmg-recomendador/main/data/dificuldade_engsis.csv

print("Dados baixados com sucesso e salvos na pasta 'data/'!")

### 2. Definição das Funções do Recomendador (Substituindo Módulos Externos)
Abaixo estão todas as funções consolidadas para:
1. Carregar a Base Curada.
2. Extrair o contexto semântico.
3. Utilizar o **Sentence-BERT** para realizar busca.
4. Aplicar regras de **Pré-Requisitos** e **Limite de Dificuldade** da UFMG.

In [ ]:
import pandas as pd
import numpy as np
import torch
import json
import warnings
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings('ignore')

# ----------------- CONFIGURAÇÕES -----------------
DISCIPLINAS_JSON = "data/disciplinas_engsis.json"
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

# ----------------- CURADORIA DE DADOS -----------------
def get_discipline_table() -> pd.DataFrame:
    with open(DISCIPLINAS_JSON, 'r', encoding='utf-8') as f:
        return pd.DataFrame(json.load(f))

def get_discipline_text(row: pd.Series) -> str:
    nome = row.get("nome", "")
    area = row.get("area_conhecimento", "")
    ementa = row.get("ementa", "")
    return f"{nome}. Área: {area}. {ementa}"

def get_completed_profile_text(df: pd.DataFrame, completed_codes: list) -> str:
    text_parts = []
    for code in completed_codes:
        disc = df[df["codigo"] == code]
        if not disc.empty:
            text_parts.append(f"{disc.iloc[0]['nome']} ({disc.iloc[0]['area_conhecimento']})")
    return "Disciplinas cursadas com interesse: " + ", ".join(text_parts) + "."

# ----------------- SBERT ENCODER -----------------
def load_sbert_model() -> SentenceTransformer:
    return SentenceTransformer(MODEL_NAME)

def encode_texts(model, texts: list, batch_size=32) -> np.ndarray:
    return model.encode(texts, batch_size=batch_size, show_progress_bar=True, convert_to_numpy=True)

def encode_disciplines(model, df: pd.DataFrame) -> np.ndarray:
    texts = df.apply(get_discipline_text, axis=1).tolist()
    return encode_texts(model, texts)

def semantic_search(query_embedding: np.ndarray, corpus_embeddings: np.ndarray, top_k=10):
    query_tensor = torch.tensor(query_embedding)
    corpus_tensor = torch.tensor(corpus_embeddings)
    hits = util.semantic_search(query_tensor, corpus_tensor, top_k=top_k)[0]
    for hit in hits:
        hit['index'] = hit.pop('corpus_id')
    return hits

# ----------------- REGRAS DE RECOMENDAÇÃO -----------------
def prerequisites_satisfied(candidate_row, completed_codes) -> bool:
    prerequisites = set(candidate_row.get("pre_requisitos", []) or [])
    return prerequisites.issubset(set(completed_codes))

def filter_recommended_candidates(candidates: pd.DataFrame, completed_codes) -> pd.DataFrame:
    if candidates.empty: return candidates.copy()
    mask = candidates.apply(lambda row: prerequisites_satisfied(row, completed_codes), axis=1)
    return candidates.loc[mask].copy()

def greedy_balance_by_difficulty(candidates: pd.DataFrame, difficulty_budget: float, max_courses=None) -> pd.DataFrame:
    if candidates.empty: return candidates.copy()
    ordered = candidates.sort_values(["predicted_probability", "difficulty"], ascending=[False, True]).copy()
    
    selected_rows = []
    used_difficulty = 0.0
    for _, row in ordered.iterrows():
        difficulty = row.get("difficulty")
        numeric_diff = float(difficulty) if pd.notna(difficulty) else 0.0
        if max_courses is not None and len(selected_rows) >= max_courses: break
        if used_difficulty + numeric_diff > difficulty_budget: continue
        selected_rows.append(row)
        used_difficulty += numeric_diff
        
    return pd.DataFrame(selected_rows).reset_index(drop=True) if selected_rows else ordered.head(0).copy()

print("Todas as funções declaradas!")

### 3. Pipeline de Execução do Modelo SBERT
Abaixo carregamos a base, fazemos o fine-tuning semântico simulado (geração de tensores) e simulamos o input.

In [ ]:
print("Carregando Base de Dados e Modelo SBERT...")
df_disciplinas = get_discipline_table()
model = load_sbert_model()

print("\nGerando Embeddings das disciplinas...")
corpus_embeddings = encode_disciplines(model, df_disciplinas)

# Simulação do Estudante
disciplinas_concluidas = ['DCC203', 'DCC204', 'MAT001', 'MAT039']
perfil_texto = get_completed_profile_text(df_disciplinas, disciplinas_concluidas)
query_texto = perfil_texto + " Quero me aprofundar em análise de dados, inteligência artificial e estatística. Tenho interesse em modelagem."
print(f"\nInput Natural do Aluno:\n{query_texto}\n")

query_embedding = encode_texts(model, [query_texto], batch_size=1)[0]
hits = semantic_search(query_embedding, corpus_embeddings, top_k=15)

candidatas_list = []
for hit in hits:
    idx = hit['index']
    row = df_disciplinas.iloc[idx].copy()
    row['predicted_probability'] = hit['score']
    row['difficulty'] = row['dificuldade_geral']
    candidatas_list.append(row)
df_candidatas = pd.DataFrame(candidatas_list)

df_validas = filter_recommended_candidates(df_candidatas, disciplinas_concluidas)
df_validas = df_validas[~df_validas['codigo'].isin(disciplinas_concluidas)]
df_recomendacao_final = greedy_balance_by_difficulty(df_validas, difficulty_budget=25.0, max_courses=4)

print("==== RECOMENDAÇÃO FINAL DO SEMESTRE ====")
for _, row in df_recomendacao_final.iterrows():
    print(f"-> {row['codigo']} - {row['nome']} (Período: {row['periodo_sugerido']})")
    print(f"   Dificuldade: {row['difficulty']} | Score Semântico: {row['predicted_probability']:.4f}\n")


### 4. Extra: Fine-Tuning do Sentence-BERT (Treinamento)
Como documentado no Artigo e nos requisitos do Trabalho Prático (TP3), utilizamos Transferência de Aprendizado. O modelo SBERT foi pré-treinado no idioma Português, mas pode passar por um **Fine-Tuning** usando os dados locais da UFMG.

Para isso, geramos **Perfis Sintéticos** combinando ementas passadas e rotulamos a relevância de matérias futuras (0 ou 1) de acordo com os pré-requisitos do fluxograma. O modelo então otimiza a perda da similaridade (`CosineSimilarityLoss`).

Abaixo demonstramos o código de treino. *(Nota: treinar o SBERT na CPU pode levar bastante tempo, então a execução desta célula é opcional).*

In [ ]:
from sentence_transformers import InputExample, losses
from torch.utils.data import DataLoader

# 1. Função para gerar pares sintéticos de treino (Perfil -> Disciplina)
def generate_sbert_training_pairs(df_disciplinas):
    """Simula pares de (Texto A, Texto B, Relevância) para treinar o SBERT"""
    records = []
    # Simulando um perfil de 1º período indo para o 2º
    texto_perfil_basico = "CÁLCULO DIFERENCIAL E INTEGRAL I (Matemática). INTRODUÇÃO A ENGENHARIA DE SISTEMAS (Engenharia)"
    
    # Exemplo Positivo (Disciplina Relevante e sem barreiras)
    records.append({
        "text_a": texto_perfil_basico,
        "text_b": "CÁLCULO DIFERENCIAL E INTEGRAL II. Área: Matemática. Integrais múltiplas, campos vetoriais.",
        "label": 1.0 # Relevante
    })
    # Exemplo Negativo (Disciplina que o aluno não tem base para fazer)
    records.append({
        "text_a": texto_perfil_basico,
        "text_b": "APRENDIZADO DE MÁQUINA. Área: Computação. Redes Neurais e IA.",
        "label": 0.0 # Irrelevante (fora do período)
    })
    return pd.DataFrame(records)

print("Gerando pares sintéticos de treino...")
df_treino = generate_sbert_training_pairs(df_disciplinas)

# 2. Convertendo para o formato do Sentence-Transformers
train_examples = []
for _, row in df_treino.iterrows():
    train_examples.append(InputExample(texts=[row['text_a'], row['text_b']], label=row['label']))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=2)
train_loss = losses.CosineSimilarityLoss(model)

# 3. Fine-Tuning do Modelo SBERT
print("Iniciando Fine-Tuning (Transfer Learning) do SBERT...")
# model.fit(
#     train_objectives=[(train_dataloader, train_loss)],
#     epochs=1, # No caso real, usar 3 a 5 épocas
#     warmup_steps=10,
#     show_progress_bar=True
# )
print("Treinamento finalizado com sucesso! (Célula comentada para não travar a CPU local).")
